# Post-quantum TLS, end to end
## Containers, router CLI, Ansible

Is the router's own HTTPS server post-quantum? On IOS XE 26.2 it already is, out
of the box. This notebook shows you that, then makes you *prove* it: first
between two OpenSSL containers, then from your laptop to a real router, then on
every router at once with Ansible.

**First time?** Build your lab with [`demos/README.md`](../README.md)
(prerequisites, credentials). TLS only needs each router's management address:
no underlay, no tunnels. This notebook checks the rest in
[Before each run](#preflight).

Four kinds of cell:

- **Act 1 cells** drive OpenSSL in containers and draw the handshake as it happens.
- **Laptop probes** (`tls_probe(...)`) run `openssl s_client` from *your* laptop
  against a router's HTTPS server and report what was negotiated. They only
  connect; nothing is configured.
- **Router checks** (`r1(...)`, `check()`) run read-only `show` commands through
  Ansible. In Act 2 you type every change yourself, and the checks confirm you
  got the result. The one exception is `setup()`, which configures and says so.
- **Playbook cells** (`playbook(...)`) run Act 3 from `deploy/c8000/automation`
  with its own venv and keep the `PLAY RECAP` visible.

## Contents

- [Setup](#setup)
- [Before each run](#preflight): containers, routers, automation
- [Act 1: containers](#act1): Demo 1 and Demo 2
- [Starting at Act 2?](#act2-setup): set up the routers without the containers
- [Act 2: the router's HTTPS server, from your laptop](#act2): Demos 3 to 5
- [Act 3: Ansible](#act3): Demo 6
- [Clean up](#after)

<a id="setup"></a>
## Setup

Run this once per kernel. It finds `pqc_demo.py` in `demos/containers_demo/`, the
shared helpers in `demos/common/` and `tls_lab.py` next to this notebook,
whichever folder Jupyter started in.

In [ ]:
import sys, pathlib, re
_here = pathlib.Path.cwd()
_demos = next((p for p in (_here.parent, _here, _here / 'demos') if (p / 'common' / 'lab.py').exists()), None)
if _demos is None:
    raise RuntimeError('Start Jupyter inside the repo (see demos/README.md) so the helpers can be found.')
for _p in ('containers_demo', 'common', 'tls_demo'):
    sys.path.insert(0, str(_demos / _p))
from pqc_demo import *  # noqa: F401,F403  containers, panels
from tls_lab import *   # noqa: F401,F403  routers, playbooks, probes, labs

# Ladders for the Act 1 panels (same objects the learner notebook uses).
tls = TLSDemo(); tls_auth = TLSAuthDemo()

set_presenter_mode(True)
print('Repo:       ', REPO_ROOT)
print('Automation: ', AUTOMATION_DIR)
print('Containers:  tls_server tls_client | mtls_server mtls_client | host')
print('Routers:     r1 r2 r3 (read-only)   Laptop: tls_probe()   Playbooks: playbook()')

---
<a id="preflight"></a>
## Before each run

Most failures are leftovers from the last run, so check everything before you
start. If a check fails, the fix is in the table below or in
the [one-time setup](../README.md).

### Containers

**First time on this machine only:** build this demo's labs. Two small images on Debian's OpenSSL 3.5, so it's quick.

In [ ]:
# host: build the Act 1 images (first time on this machine)
labs.build()

In [ ]:
# host: bring this demo's stacks up, then show the readiness board
labs.up()

If anything in Act 1 sticks later, `labs.panic()` stops stray daemons and
captures without taking the stacks down.

The board below should be all green. No Docker? The container rows fail, and
that's fine if you start at Act 2.

In [ ]:
# host: Act 1 containers up, Act 2/3 prerequisites in place
labs.status()

The last row is your laptop's OpenSSL. Acts 2 and 3 run `openssl s_client` from
here, and anything older than 3.5 has never heard of `X25519MLKEM768`.

### Routers: is the lab in Act 2's starting state?

The target: **every router's HTTPS server is on, at the 26.2 default (`PQC type:
all`), with no `ip http secure-pqc-type` line, serving a classical certificate.**
A router that carries the IPsec demo's ML-DSA trustpoint also needs its HTTPS
trustpoint pinned; Demo 4 explains why.

This cell only reads (about 15 seconds). Starting from scratch?
[`setup()`](#act2-setup) fixes every row except a server that's off.

In [ ]:
# R1, R2, R3: read-only checks of the Act 2 starting state
check()

**If a row fails**, here's where the fix lives:

| Row | Fix |
|---|---|
| HTTPS server | `ip http secure-server`. It needs a certificate; the router's own self-signed one is fine here |
| PQC type, pqc-type line | [`setup()`](#act2-setup), or `no ip http secure-pqc-type` |
| classical certificate, trustpoint pinned | [`setup()`](#act2-setup), or `ip http secure-trustpoint <your self-signed trustpoint>` (`show crypto pki trustpoints` lists them) |

Want the raw output behind a row? Ask the router directly:

In [ ]:
# R1: any single read-only command, real output
r1('show ip http server secure status');

### Automation

`verify.yml` only reads (NETCONF and `show`), so it's safe at any time. It proves
the playbooks can reach all three routers and prints their crypto posture.

In [ ]:
# Ansible: read-only posture report across r1, r2, r3
playbook('verify.yml');

**`Could not open socket to <router>:830`?** NETCONF isn't on. `bootstrap.yml`
turns it on (that's a config change, so it's not part of the read-only checks)
and waits for port 830 to answer. Run it, then rerun `verify.yml`.

In [ ]:
# Ansible: enable NETCONF on r1, r2, r3 (only if verify.yml can't reach port 830)
playbook('ansible-playbook bootstrap.yml');

---
<a id="act1"></a>
## Act 1: learn it in containers

An OpenSSL server and client on one Docker network, with `tcpdump` on the
server. One cell per action: read the comment, run it, read the output.

### Demo 1: hybrid key exchange, no extra round trip

What does ML-KEM change in a TLS 1.3 handshake? Look for:

- `Negotiated TLS1.3 group: X25519MLKEM768`
- key shares of **1216 B** in the ClientHello and **1120 B** in the ServerHello
- the classical contrast: **32 B** each way, same single round trip

So the cost is bytes, not a new flight. Keep those two numbers in mind; Act 2
measures them again on the router.

**Beat 1: start the hybrid TLS 1.3 server + capture**

In [ ]:
# TLS server: mint a throwaway EC P-256 cert for the listener
tls_server("openssl genpkey -algorithm EC -pkeyopt ec_paramgen_curve:P-256 -out /tmp/server.key 2>/dev/null; echo key-ready")
tls_server("openssl req -x509 -new -key /tmp/server.key -out /tmp/server.crt -days 365 -subj /CN=tls-server.pqc.lab 2>/dev/null; echo cert-ready");

In [ ]:
# TLS server: packet capture, then a hybrid TLS 1.3 listener
tls_server("pkill -f [s]_server 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true")
tls_server("tcpdump -i any -U 'tcp port 4433' -w /tmp/hybrid.pcap &")
tls_server("openssl s_server -accept 4433 -cert /tmp/server.crt -key /tmp/server.key -tls1_3 -groups X25519MLKEM768 -www &");

**Beat 2: client connects, prints the negotiated group**

In [ ]:
# TLS client: connect and print the negotiated TLS 1.3 group
neg = tls_client("openssl s_client -connect 172.22.0.2:4433 -groups X25519MLKEM768 </dev/null 2>&1 | grep -iE 'Negotiated TLS1.3 group|Protocol|Cipher is'")

In [ ]:
panel(tls.stage, step=1, focus=["ClientHello", "ServerHello"], status=("active", "ok"),
      rows=[("Negotiated TLS 1.3 group", "X25519MLKEM768",
             (grep(neg.text, 'Negotiated') or ['?'])[0], 'MLKEM' in neg.text.upper())],
      banner="One round trip: the group is chosen in the ClientHello/ServerHello, same as classical.")

**Beat 3: read the ML-KEM key-share sizes off the capture, client vs. server**

In [ ]:
# TLS server: tcpdump needs a moment to drain its buffer before we stop it
tls_server("pkill -f [s]_server 2>/dev/null; sleep 2; pkill -f [t]cpdump 2>/dev/null; sleep 1; true");

In [ ]:
# TLS server: split the capture into the ClientHello vs. ServerHello key share
client_share = tls_server("tshark -r /tmp/hybrid.pcap -V 2>/dev/null | awk '/Handshake Type: Client Hello/{f=1} /Handshake Type: Server Hello/{f=0} f' | grep -i 'Key Share Entry: Group:' | head -1")
server_share = tls_server("tshark -r /tmp/hybrid.pcap -V 2>/dev/null | awk '/Handshake Type: Server Hello/{f=1} f' | grep -i 'Key Share Entry: Group:' | head -1")

In [ ]:
panel(tls.stage, step=3, focus=["ClientHello", "ServerHello"], status=("ok", "ok"),
      rows=[("Client -> Server key share (in ClientHello)", "1216 B", kex_len(client_share.text) + " B",
             kex_len(client_share.text) == "1216"),
            ("Server -> Client key share (in ServerHello)", "1120 B", kex_len(server_share.text) + " B",
             kex_len(server_share.text) == "1120")],
      notes=[("Round trips", "1 (no extra flight)")],
      banner="Hybrid key exchange with no extra round trip: the cost is just bytes, not a new flight.")

**Beat 4 (optional): classical x25519 for contrast (32 B shares, still 1-RTT)**

In [ ]:
# TLS server: restart the listener classical-only (x25519)
tls_server("pkill -f [s]_server 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true")
tls_server("tcpdump -i any -U 'tcp port 4433' -w /tmp/classical.pcap &")
tls_server("openssl s_server -accept 4433 -cert /tmp/server.crt -key /tmp/server.key -tls1_3 -groups x25519 -www &");

In [ ]:
# TLS client: connect classical-only
tls_client("openssl s_client -connect 172.22.0.2:4433 -groups x25519 </dev/null 2>&1 | grep -iE 'Negotiated TLS1.3 group'");

In [ ]:
# TLS server: drain, stop, and read the classical key-share size
tls_server("pkill -f [s]_server 2>/dev/null; sleep 2; pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
classical_share = tls_server("tshark -r /tmp/classical.pcap -V 2>/dev/null | grep -i 'Key Share Entry: Group:' | head -1")

In [ ]:
# Retag the same two arrows for this contrast run; they said '...hybrid share'
# a moment ago; x25519-only shares are the same 32 B on both sides
classical_tag = kex_len(classical_share.text) + ' B classical share'
tls.stage.retag("ClientHello", classical_tag, direction="r")
tls.stage.retag("ServerHello", classical_tag, direction="l")
panel(tls.stage, step=1, focus=["ClientHello", "ServerHello"], status=("ok", "ok"),
      rows=[("Classical key share (each side)", "32 B", kex_len(classical_share.text) + " B",
             kex_len(classical_share.text) == "32")],
      banner="Classical x25519: 32-byte shares, same single round trip. The hybrid cost is purely bytes.",
      banner_color=SLATE)

In [ ]:
# reset (stop the TLS listener + capture)
tls_server("pkill -f [s]_server 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true");

### Demo 2: post-quantum certificates

The key exchange is post-quantum; the certificate signature isn't. What happens
when it is? Look for:

- the DER sizes: ML-DSA-65 is about **15x** the ECDSA leaf
- mutual TLS with ECDSA: `ecdsa_secp256r1_sha256`, about **2.5 KB** read
- the same handshake with ML-DSA-65: `mldsa65`, about **16 KB** read

On TCP that's just a longer handshake, no fragmentation to manage (unlike IKEv2).

**Beat 1: weigh the certificates (ML-DSA-65 is ~15x the ECDSA leaf)**

In [ ]:
# TLS server: mint one cert per algorithm and list DER sizes
mtls_server("cd /tmp && for e in EC:ecdsa ML-DSA-44:mldsa44 ML-DSA-65:mldsa65 ML-DSA-87:mldsa87; do a=${e%%:*}; n=${e##*:}; if [ $a = EC ]; then openssl genpkey -algorithm EC -pkeyopt ec_paramgen_curve:P-256 -out $n.key 2>/dev/null; else openssl genpkey -algorithm $a -out $n.key 2>/dev/null; fi; openssl req -x509 -new -key $n.key -out $n.crt -days 365 -subj /CN=$n -outform DER 2>/dev/null; done; ls -l *.crt | awk '{print $5, $9}' | sort -n");

**Beat 2: mutual TLS with classical ECDSA certificates**

In [ ]:
# host: mint an ECDSA CA + server/client certs
host("cd learn/tls/authentication && docker compose run --rm certgen ecdsa");

In [ ]:
# TLS server: require + verify a client certificate (mutual TLS)
mtls_server("pkill -f [s]_server 2>/dev/null; true")
mtls_server("openssl s_server -accept 4433 -cert /cfg/server.crt -key /cfg/server.key -CAfile /cfg/ca.crt -Verify 1 -tls1_3 -www &");

In [ ]:
# TLS client: present its own certificate and connect
res = mtls_client("openssl s_client -connect 172.23.0.2:4433 -cert /cfg/client.crt -key /cfg/client.key -CAfile /cfg/ca.crt </dev/null 2>&1 | grep -iE 'Verify return code|Peer signature type|SSL handshake has read'")
mtls_server("pkill -f [s]_server 2>/dev/null; true")
reads = first_int((grep(res.text, 'has read') or [''])[0])

In [ ]:
panel(tls_auth.stage, step=3,
      focus=["ServerHello + Certificate", "Client Certificate"], status=("ok", "ok"),
      rows=[("Peer signature type", "ecdsa_secp256r1_sha256", (grep(res.text, 'signature type') or ['?'])[0], 'ecdsa' in res.text.lower()),
            ("Verify return code", "0 (ok)", (grep(res.text, 'Verify return') or ['?'])[0], '0 (ok)' in res.text)],
      notes=[("Handshake bytes read", reads + ' B (~2.5 KB)')],
      banner="Baseline: mutual TLS with classical ECDSA certs over a hybrid ML-KEM key exchange.")

**Beat 3: same handshake with ML-DSA-65 certificates**

In [ ]:
# host: reissue the CA + certs as ML-DSA-65
host("cd learn/tls/authentication && docker compose run --rm certgen ml-dsa-65");

In [ ]:
# TLS server + client: same mutual-TLS handshake, new certificates
mtls_server("pkill -f [s]_server 2>/dev/null; true")
mtls_server("openssl s_server -accept 4433 -cert /cfg/server.crt -key /cfg/server.key -CAfile /cfg/ca.crt -Verify 1 -tls1_3 -www &")
res = mtls_client("openssl s_client -connect 172.23.0.2:4433 -cert /cfg/client.crt -key /cfg/client.key -CAfile /cfg/ca.crt </dev/null 2>&1 | grep -iE 'Verify return code|Peer signature type|SSL handshake has read'")
mtls_server("pkill -f [s]_server 2>/dev/null; true")
reads = first_int((grep(res.text, 'has read') or [''])[0])

In [ ]:
panel(tls_auth.stage, step=3,
      focus=["ServerHello + Certificate", "Client Certificate"], status=("ok", "ok"),
      rows=[("Peer signature type", "mldsa65", (grep(res.text, 'signature type') or ['?'])[0], 'mldsa65' in res.text.lower()),
            ("Verify return code", "0 (ok)", (grep(res.text, 'Verify return') or ['?'])[0], '0 (ok)' in res.text)],
      notes=[("Handshake bytes read", reads + ' B (~16 KB)')],
      banner="Same handshake, ML-DSA-65 certs: ~2.5 KB -> ~16 KB. Both halves now quantum-safe (ML-KEM + ML-DSA).",
      banner_color=MAGENTA)

Something stuck? This stops stray listeners and captures without taking the
stacks down:

In [ ]:
# host: stop stray s_server / tcpdump processes, keep the stacks up
labs.panic()

---
<a id="act2-setup"></a>
## Starting at Act 2? Set up the routers

Skipping the containers? Start here. This cell loads its own helpers, so you
don't need the Setup cell at the top. Then, on each router:

1. removes any `ip http secure-pqc-type` line, back to the default (`all`)
2. if the router carries the ML-DSA trustpoint `TP-MLDSA65`, pins the HTTPS
   server to its classical self-signed trustpoint (Demo 4 says why)
3. NETCONF on all three, for Act 3
4. the same read-only check as the pre-flight

**This cell configures the routers**, unlike every other router cell here. It
never turns the HTTPS server on for you: that needs a certificate decision, and
that's yours. On a lab that's already in shape it only reads.

In [ ]:
# R1, R2, R3: CONFIGURES the Act 2 starting state, then checks it
# (loads the helpers first, so this works as the first cell you run)
import sys, pathlib
_here = pathlib.Path.cwd()
_demos = next((p for p in (_here.parent, _here, _here / 'demos') if (p / 'common' / 'lab.py').exists()), None)
if _demos is None:
    raise RuntimeError('Start Jupyter inside the repo (see demos/README.md) so the helpers can be found.')
for _p in ('containers_demo', 'common', 'tls_demo'):
    sys.path.insert(0, str(_demos / _p))
from tls_lab import *  # noqa: F401,F403  routers, playbooks, probes

setup()

---
<a id="act2"></a>
## Act 2: the router's HTTPS server, from your laptop

**You type every router command in this act at the R1 CLI.** The probes run
`openssl s_client` from your laptop to R1's management address on port 443, so
what you see is what any TLS 1.3 client gets from that router.

### Demo 3: it's already post-quantum

**R1:**

```
show ip http server secure status
```

Look for `HTTP secure server PQC type: all` (new in 26.2, and the reason this works
with no config) and `HTTP secure server ECDHE curve: secp384r1` (that one comes back
in Demo 4). Now ask from the laptop:

In [ ]:
# laptop -> R1:443: what does a default OpenSSL client negotiate?
tls_probe('r1');

`Negotiated TLS1.3 group: X25519MLKEM768` is the whole result: the same hybrid
group the containers agreed on, now spoken by the router. Then read
`Signature type: rsa_pss_rsae_sha256`. The key exchange is post-quantum, the
certificate is still classical RSA, and on 26.2 the HTTPS server can't serve an
ML-DSA one.

Same bytes as Act 1? Read them off the handshake:

In [ ]:
# laptop -> R1:443: key_share sizes per hello (openssl s_client -trace)
tls_probe('r1', trace=True);

Measured on IOS XE 26.2: the ClientHello's `key_share` extension is **1258 B**
(OpenSSL offers two shares: 1216 B of X25519MLKEM768 plus 32 B of x25519, plus
headers). The ServerHello's is **1124 B**: the 1120 B share from Act 1, plus its
header. Same numbers, real router.

### Demo 4: steer the key exchange

`all` isn't the only setting. Force the server classical. **R1:**

```
conf t
 ip http secure-pqc-type non-pqc
end
```

**Heads up: pin the HTTPS trustpoint first.** Changing this knob reloads the
HTTPS server, and an unpinned server picks its certificate again. On our R1, which
carries the IPsec demo's ML-DSA trustpoint, it picked `TP-MLDSA65`, which 26.2
can't serve: every handshake then died with `alert internal error`. That's why
`check()` wants `ip http secure-trustpoint` on any router with `TP-MLDSA65`, and
why `setup()` sets it.

Now connect again with the same client offering the same groups:

In [ ]:
# laptop -> R1:443: same client, the server now refuses both offered shares
tls_probe('r1', trace=True, expect='secp384r1');

Read that exchange carefully, because it's a textbook **HelloRetryRequest**. The
client offers X25519MLKEM768 and x25519. The server wants neither, and answers
with a bare 2-byte `key_share` naming `secp384r1` (the `ECDHE curve` from Demo 3).
The client retries with a 103 B P-384 share, and the handshake finishes
classically. That retry is a whole extra round trip, the price of a server that
won't take what the client offered first.

In [ ]:
# R1: the knob is set (read-only)
router_check('Demo 4: classical only', 'R1', [
    Check('r1', 'show ip http server secure status', 'PQC type', 'non-pqc', r'PQC type: non-pqc'),
    Check('r1', 'show running-config | include ^ip http secure-', 'config line', 'ip http secure-pqc-type non-pqc', r'secure-pqc-type non-pqc'),
    Check('r1', 'show ip http server secure status', 'still a classical certificate', 'Self or TP-self-signed-...', r'server trustpoint: (?!TP-MLDSA65\b)\S+'),
])

### Demo 5: fail closed

`non-pqc` falls back. What if you want the opposite: refuse anyone who isn't
post-quantum? **R1:**

```
conf t
 ip http secure-pqc-type pqc
end
```

Then try the same default client:

In [ ]:
# laptop -> R1:443: the default client offers hybrid and x25519
tls_probe('r1', expect='refused');

**It fails** (`alert handshake failure`), even though the client offered a hybrid
post-quantum group. `pqc` means *pure* ML-KEM, and OpenSSL's default list doesn't
offer that. Ask for it explicitly:

In [ ]:
# laptop -> R1:443: offer pure ML-KEM-768 only
tls_probe('r1', groups='MLKEM768', expect='MLKEM768');

Here's all four settings against four client offers, measured from an OpenSSL 3.6
client against R1 (`tls_probe('r1', groups=...)` reproduces any cell):

| `ip http secure-pqc-type` | default client | `X25519` only | `secp384r1` only | `MLKEM768` only |
|---|---|---|---|---|
| `all` (the default) | X25519MLKEM768 | refused | secp384r1 | MLKEM768 |
| `hybrid` | X25519MLKEM768 | refused | refused | refused |
| `non-pqc` | secp384r1, after a HelloRetryRequest | refused | secp384r1 | refused |
| `pqc` | refused | refused | refused | MLKEM768 |

Notice the `X25519` column: refused every time. The server's only classical group
is its `ECDHE curve`, `secp384r1`, so a client that offers nothing but x25519 never
gets in, whatever the PQC setting.

When would you set one? `non-pqc` for a client that chokes on a big ClientHello,
`hybrid` or `pqc` to *require* post-quantum and fail closed. `all` is the sane
default, and it's what ships.

**Put it back on R1.** Either line works; `all` is the default, so neither one
leaves anything in the running config:

```
conf t
 no ip http secure-pqc-type
end
```

In [ ]:
# laptop -> R1:443: back to the hybrid default
tls_probe('r1');

---
<a id="act3"></a>
## Act 3: now do it to the fabric

One router by hand is fine. What about fifty? `ip http secure-pqc-type` is one
modelled YANG leaf, so `tls-pq.yml` is the smallest playbook in the automation
layer: one NETCONF edit per router, no CLI at all.

Run `check()` first if you skipped Act 2: the playbook reloads every HTTPS
server, and the Demo 4 trap applies to all of them.

### Demo 6: Ansible states the posture, then proves the knob

Run it twice. The first run writes `secure-pqc-type all` on every router and
saves; the second finds nothing to do. Why push the default? So the posture is
*stated* in config, not an accident of the release.

Captured run (`captured/09-tls-pq/`): `changed=1` on each router, then
`changed=0`. Each run takes about 40 s (37 to 41 s on 2026-10-29).

Why can the panel say 1 when the recap says 2? Every playbook ends with `verify.yml`, whose posture report is forced to `changed` so it stays on screen. The panel subtracts it and counts config changes only, which is what the captured runs show.

These cells keep the `PLAY RECAP` on screen. **Heads up:** in a plain terminal the repo's `pqc_lab` callback hides the recap when a run succeeds. Run them from here, or prefix `ANSIBLE_STDOUT_CALLBACK=default` in a terminal (plus `-e @../../../demos/common/lab.yml`; see [Running a playbook from a terminal](../README.md#run-a-playbook-from-a-terminal)).

In [ ]:
# Ansible: state the HTTPS posture (pq_https_pqc_type, default 'all') on r1, r2, r3
playbook('ansible-playbook tls-pq.yml', expect_changed=1);

In [ ]:
# Ansible: same playbook again; nothing to do is the point
playbook('ansible-playbook tls-pq.yml', expect_changed=0);

In [ ]:
# laptop -> R1, R2, R3: every router negotiates the hybrid group
probe_all();

Now prove the knob does something, fleet-wide. The playbook reads
`pq_https_pqc_type` from `group_vars/all/pqc.yml`; override it for one run:

In [ ]:
# Ansible: force every HTTPS server classical
playbook('ansible-playbook tls-pq.yml -e pq_https_pqc_type=non-pqc', expect_changed=1);

In [ ]:
# laptop -> R1, R2, R3: HelloRetryRequest to secp384r1 everywhere
probe_all(expect='secp384r1', trace=True);

---
<a id="after"></a>
## Clean up

Remove the leaf on all three routers. Teardown deletes it rather than writing a
value, so each box goes back to its own default (`all`):

In [ ]:
# Ansible: remove ip http secure-pqc-type from r1, r2, r3
playbook('ansible-playbook tls-pq.yml -e state=absent', expect_changed=1);

In [ ]:
# R1, R2, R3: back in the starting state, and negotiating hybrid again
check() and probe_all();

The pinned `ip http secure-trustpoint` stays. It's what keeps the next restart
from picking the ML-DSA trustpoint, so leave it unless that trustpoint is gone.

On the laptop: containers down, generated certificates removed, `learn/tls` back
to the committed state.

In [ ]:
# host: Act 1 stacks down, generated keys and certs removed, learn/tls back to the committed state
labs.cleanup()

**Run it again?** Start from [Before each run](#preflight). Clean up leaves the
routers in the starting state, so `check()` should be green straight away.